In [2]:
import pandas as pd
import pathlib
import os

os.chdir("../..")

# Sorry for this shitty path variables, It will be fixed on server soon (fix for this error: "Permission denied: '/root/.cargo/bin'")
os.environ["PATH"] = ":".join(
    p for p in os.environ["PATH"].split(":")
    if not p.startswith("/root")
)


import pymaid

catmaid_url = 'https://l1em.catmaid.virtualflybrain.org'
http_user = None
http_password = None
project_id = 1

rm = pymaid.CatmaidInstance(catmaid_url, http_user, http_password, project_id)

INFO  : Global CATMAID instance set. Caching is ON. (pymaid)


In [3]:
# -------------------- get ALL neuron ids --------------------
neuron_ids = sorted(map(int, pymaid.get_skids_by_annotation('mw brain and inputs')))

len(neuron_ids)

INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)


3016

# Use Cell Type Synapse Mapping

In [6]:
s2 = pd.read_csv('./Datasets/Original/s2.csv')
s3 = pd.read_csv('./Datasets/Original/s3.csv')
s4 = pd.read_csv('./Datasets/Original/s4.csv')
out = pd.read_csv('./Datasets/Original/outputs.csv', index_col=0)


# -------------------- load attributes --------------------
df = pd.DataFrame({'neuron_id': neuron_ids})
attr = s3.merge(
    s4,
    on='skid',
    how='left',
    suffixes=('_axon', '_dendrite')
).rename(columns={'skid': 'neuron_id'})

# merge attributes onto full neuron list
df = df.merge(attr, on='neuron_id', how='left')

# -------------------- cell_type (RAW, no mapping) --------------------
cell_types = []

for _, row in df.iterrows():
    ct = row.get('celltype_axon')
    if pd.isna(ct):
        ct = row.get('celltype')
    cell_types.append(ct if pd.notna(ct) else None)

df['cell_type'] = cell_types

# -------------------- synapse_type --------------------
CELLTYPE_MAP = {
    'KCs': 'exc', 
    'PNs': 'exc', 
    'PNs-somato': 'exc',
    'LNs': 'inh', 
    'MB-FBNs': 'inh', 
    'MB-FFNs': 'inh',
    'pre-DN-SEZs': 'mixed', 
    'pre-DN-VNCs': 'mixed',
    'RGNs': 'inh', 
    'DN-VNCs': 'mixed',
    'LHNs': 'exc', 
    'MBONs': 'exc',
    'MBINs': 'mod', 
    'DN-SEZs': 'mixed', 
    'CNs': 'mixed'
}

synapse_types = []

for ct in df['cell_type']:
    if ct is None:
        synapse_types.append(None)
    else:
        mapped = CELLTYPE_MAP.get(ct)
        if mapped == 'inh':
            synapse_types.append('inhibitory')
        elif mapped == 'exc':
            synapse_types.append('excitatory')
        elif mapped in ('mixed', 'mod'):
            synapse_types.append('mixed')
        else:
            synapse_types.append(None)

df['synapse_type'] = synapse_types

# -------------------- load INPUT neurons --------------------
input_rows = s2[s2['celltype'].isin(['sensory', 'ascending'])]

left_ids = pd.to_numeric(input_rows['left_id'], errors='coerce').dropna()
right_ids = pd.to_numeric(input_rows['right_id'], errors='coerce').dropna()

input_neurons = set(left_ids.astype(int)).union(set(right_ids.astype(int)))

# -------------------- load OUTPUT neurons --------------------
output_neurons = set(out[out['axon_output'] > 50].index.astype(int))

# -------------------- IO column --------------------
IO = []

for nid in df['neuron_id']:
    if nid in input_neurons:
        IO.append('input')
    elif nid in output_neurons:
        IO.append('output')
    else:
        IO.append(None)

df['IO'] = IO

# -------------------- final table --------------------
df = df[['neuron_id', 'cell_type', 'synapse_type', 'IO']]

df.to_csv("./Datasets/Generated/Syn_Types(By_Cell_Types).csv")

df

,neuron_id,cell_type,synapse_type,IO
0,29,KCs,excitatory,output
1,37365,None,None,input
2,40045,None,None,input
3,40152,None,None,input
4,677717,KCs,excitatory,output
...,...,...,...,...
3011,21590978,None,None,output
3012,21591033,pre-DN-VNCs,mixed,output
3013,21591037,pre-DN-VNCs,mixed,output
3014,21591317,None,None,None


# Use Catmaid Annotation Mapping
Synapse types will be classified by LLM

In [15]:
# metadata = pd.read_pickle("./Datasets/Original/Metadata(auto).pkl")
# metadata.set_index("skeleton_id", inplace=True)
# metadata.index = metadata.index.astype(int)

# metadata_3k = metadata[metadata.index.isin(neuron_ids)]
# metadata_3k

In [22]:
neurons_metadata = pd.read_csv("./Datasets/Generated/Metadata_Neurons(manual).csv")
neurons_metadata

,neuron_id,name,type,n_nodes,n_connectors,n_branches,n_leafs,cable_length,annotation
0,7766016,H-shaped _a1l,CatmaidNeuron,5745,374,138,141,620329.900,"['YYCWMLEH', '4_level-4_clusterID-22_signal-fl..."
1,19431430,SOG IN left; PN into SOG very minor AL connect...,CatmaidNeuron,3029,215,129,136,334532.600,"['RGPN_inputs_T3_sens', 'fragment', 'antennal ..."
2,15564807,AN-R-Sens-B1-AVa-22,CatmaidNeuron,800,34,7,8,69523.360,"['monosynaptic to IPC-like and Se0', 'Miroschn..."
3,17383431,BAlp_ant ascending dendrite left,CatmaidNeuron,896,109,64,70,205249.700,"['1_level-1_clusterID-1_signal-flow_-0.067', '..."
4,7782409,Drunken-4 a1l,CatmaidNeuron,2749,106,39,41,265065.560,"['CCWA09Input', 'A1L Interneurons', 'sw;idev3'..."
...,...,...,...,...,...,...,...,...,...
5008,17629170,BAlp ant s 8 Right,CatmaidNeuron,298,0,1,2,29098.191,"['AM_published', 'Andrade et al. 2019', 'ND_al..."
5009,2637812,A02n_a1r Pseudolooper-4,CatmaidNeuron,1952,106,76,81,198657.340,"['RF PMNs for Brandon', 'RF PMN_MN for Brandon..."
5010,3882998,CP contra to SEZ left; MB1: incomplete neuron ...,CatmaidNeuron,2528,58,33,34,298192.720,"['mw dSEZ', '8_level-4_clusterID-16_signal-flo..."
5011,15679484,MN-L-Sens-B2-VM-08,CatmaidNeuron,851,30,5,6,70720.140,"['Miroschnikow et al inputs and outputs', 'AM_..."


In [28]:
neurons_metadata['annotation'][15]

"['SLP*', 'AM_published', 'Check 49a PN-down', 'DPLm', 'Common dwnst partners of right pdfLNs', 'DPLm b', 'DPLm a', 'Dorsal Neurons', 'ascending_hairbrush', 'DN2', 'Brain', 'uptoDN2', 'WB_DPLm R', 'upstream to DN2', 'mw gustatory-external 3rd_order', 'mw olfactory 3rd_order', 'mw gustatory-pharyngeal 3rd_order', 'ASB LH Interneuron', 'mw dSEZ-feedback-2hop 2022-03-15', 'mw cascade-8-hop integrative 2022-11-03', 'downstream of handlebar-like', 'sw;pairs;R', 'mw dSEZ-feedback-5hop 2022-03-15', 'mw dSEZ-efference-4hop 2022-03-15', 'mw dSEZ-efference-5hop 2022-03-15', 'mw brain neurons + sensories', 'mw dVNC-feedback-casc-8hop 2022-03-15', 'olfinter nosens to PTTH and CALP', 'mw dSEZ-efference-casc-8hop 2022-03-15', 'PTTH_inputs_T5_SH', 'CALP_inputs_T5_SH', 'RGPNupstream_T5_nosens_SH', 'mw brain and inputs', '0_level-0_clusterID-0_signal-flow_-0.011', '0_level-1_clusterID-2_signal-flow_0.063', '1_level-2_clusterID-5_signal-flow_0.681', '1_level-3_clusterID-12_signal-flow_0.836', '5_level-4